# Rebuild Nature-style ISA figures from complete ISA outputs

This Google Colab notebook rebuilds the ISA figures from the completed ISA output tables rather than from already-rendered figures. It is designed as an auditable plotting document: before each panel is drawn, it prints the exact plotting data, the processing steps and the biological/statistical question being asked.

Two evidence layers are kept separate.

1. **Full-task layer:** CAGE, DEV and HK are complete ISA tasks. Each task is calibrated only against its own pseudo-site null tables.
2. **CAGE-associated motif-position overlap layer:** CAGE motif-called regions are stratified by motif-position interval overlap with DEV and/or HK. This is a grammar/architecture audit, not the same evidence type as full-task null-calibrated ISA.

## Read this first: what the three ISA quantities mean

Ep_ISA_NEW starts from motif calls in each task-specific sequence set. It then asks how the model prediction changes when motif intervals are computationally ablated. Three quantities appear repeatedly in the figures.

**1. Single-motif ISA value.** This is the model-output change after ablating one motif interval:

\[
ISA(A)=f(wt)-f(A)
\]

Here `wt` is the original sequence and `A` is the sequence after motif A is ablated. A large positive single-motif ISA means the model prediction drops when that motif is removed, so the motif interval was contributing positively to that task's model output. It does **not** mean the sequence contains only one motif.

**2. Motif-pair interaction value.** This is computed for two motif intervals in the same region by comparing the observed double-ablation effect with the additive expectation from two single ablations:

\[
I_{A,B}=f(wt)-f(A)-f(B)+f(A+B)
\]

Here `A` and `B` are single motif ablations and `A+B` is the double ablation. This is a model-inferred non-additivity score for a motif-pair instance, not a biochemical synergy assay.

**3. TF-pair class `C_effective`.** Individual motif-pair instances are grouped by motif identities, such as `DRE|HD`. Within a TF-pair class, only interactions that exceed that task's pair-null thresholds are called `effective`. The class-level direction score is:

\[
C_{A,B}=\frac{\sum I_{A,B}^{effective}}{\sum |I_{A,B}^{effective}|}
\]

`C_effective` ranges from -1 to +1. Values near -1 mean the effective interactions are mostly negative, values near +1 mean they are mostly positive, and values near 0 mean mixed or weak directional consistency. It is a direction summary, not a new ISA experiment and not proof of biochemical synergy.

**Null tables.** `null_isa.csv` and `null_interaction.csv` are task-specific empirical background tables generated by `Ep_ISA_NEW` from non-motif pseudo-sites. They answer: what score size would we expect if we ablated comparable non-motif positions rather than motif-called positions? They are loaded directly in this plotting notebook and are never pooled across CAGE, DEV and HK.

In [ ]:
# Colab/runtime setup
import os, sys, json, math, warnings, subprocess
from pathlib import Path
from dataclasses import dataclass

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "pillow"], check=False)
    from google.colab import drive
    drive.mount("/content/drive")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, spearmanr
from PIL import Image
from IPython.display import display, Markdown

# Nature-style matplotlib settings: editable SVG text, compact typography, white background.
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Liberation Sans", "DejaVu Sans"]
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
mpl.rcParams.update({
    "font.size": 7, "axes.spines.right": False, "axes.spines.top": False,
    "axes.linewidth": 0.75, "legend.frameon": False,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "figure.dpi": 150, "savefig.dpi": 600
})
if "Arial" not in {f.name for f in mpl.font_manager.fontManager.ttflist}:
    print("Arial was not detected; matplotlib will use Liberation Sans/DejaVu Sans fallback.")

In [ ]:
# User-editable paths
RESULT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/DeepEpromote/Drosophila/DeepCAGE/script/Motif_discover/Ep_ISA_NEW/result"),
    Path("/content/drive/My Drive/DeepEpromote/Drosophila/DeepCAGE/script/Motif_discover/Ep_ISA_NEW/result"),
]
if os.name == "nt":
    RESULT_ROOT_CANDIDATES.append(Path("G:/我的云端硬盘/DeepEpromote/Drosophila/DeepCAGE/script/Motif_discover/Ep_ISA_NEW/result"))

RESULT_ROOT = next((p for p in RESULT_ROOT_CANDIDATES if p.exists()), RESULT_ROOT_CANDIDATES[0])
OUT_ROOT = Path("/content/drive/MyDrive/DeepEpromote/Drosophila/DeepCAGE/script/Motif_discover/Ep_ISA_NEW/nature_rebuilt_figures_colab") if IN_COLAB else Path.cwd() / "nature_rebuilt_figures_colab"
FIG_DIR, SOURCE_DIR = OUT_ROOT / "figures", OUT_ROOT / "source_data"
FIG_DIR.mkdir(parents=True, exist_ok=True); SOURCE_DIR.mkdir(parents=True, exist_ok=True)
print("RESULT_ROOT =", RESULT_ROOT)
print("OUT_ROOT    =", OUT_ROOT)

In [ ]:
@dataclass(frozen=True)
class TaskSpec:
    label: str
    folder: str
    track: int
    color: str

TASKS = [
    TaskSpec("CAGE", "results_cage", 0, "#BA611B"),
    TaskSpec("DEV", "results_dev", 0, "#148A6A"),
    TaskSpec("HK", "results_hk", 1, "#166C9C"),
]
TASK_ORDER = [t.label for t in TASKS]
TASK_COLOR = {t.label: t.color for t in TASKS}
NULL_COLOR, REAL_COLOR = "#767676", "#272727"
SIGN_COLORS = {"Negative": "#5B7FCA", "Near-null": "#B8B8B8", "Near-zero": "#B8B8B8", "Positive": "#D9544D"}
CONTEXT_COLORS = {"Flank-sparse": "#767676", "Flank-dense": "#C9473D"}

NULL_PERCENTILE = 0.80
Q_VALUE_THRESHOLD = 0.10
MIN_EFFECTIVE_COUNT = 10
DISTANCE_BIN_SIZE = 10
STABLE_DISTANCE_BIN_N = 30
FLANK_WIDTH_BP = 50
SEQUENCE_WINDOW_END = 249
REQUIRED_FILES = ["motif_single_isa.csv", "motif_combi_isa.csv", "null_isa.csv", "null_interaction.csv", "motif_locs.csv"]

def data_dir(task): return RESULT_ROOT / task.folder / "Data"
def isa_col(task): return f"isa_t{task.track}"
def interaction_col(task): return f"interaction_t{task.track}"

missing = [str(data_dir(t) / f) for t in TASKS for f in REQUIRED_FILES if not (data_dir(t) / f).exists()]
if missing:
    print("Missing required ISA files:")
    for x in missing: print(" -", x)
    raise FileNotFoundError("Edit RESULT_ROOT above and rerun.")
print("All required complete ISA tables were found.")

In [ ]:
def read_task_table(task, name, **kwargs):
    return pd.read_csv(data_dir(task) / name, **kwargs)

def load_complete_isa_outputs():
    return {t.label: {f.replace(".csv", ""): read_task_table(t, f) for f in REQUIRED_FILES} for t in TASKS}

raw = load_complete_isa_outputs()
inventory = []
for t in TASKS:
    for key, df in raw[t.label].items():
        inventory.append({"task": t.label, "table": key, "rows": len(df), "columns": len(df.columns), "first_columns": ", ".join(df.columns[:8])})
inventory = pd.DataFrame(inventory)
inventory.to_csv(SOURCE_DIR / "00_loaded_isa_inventory.csv", index=False)
display(inventory)

In [ ]:
def panel_context(fig, panel, title, question, steps, df=None, max_rows=30):
    display(Markdown(f"### {fig}{panel}. {title}"))
    print("Question:", question)
    print("Processing steps:")
    for i, s in enumerate(steps, 1): print(f"  {i}. {s}")
    if df is not None:
        print(f"Data used for plotting: shape={df.shape}")
        display(df.head(max_rows) if len(df) > max_rows else df)

def add_panel_label(ax, label, x=-0.08, y=1.05):
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=9, fontweight="bold")

def save_figure(fig, stem):
    paths = []
    for ext in ["png", "svg", "pdf"]:
        p = FIG_DIR / f"{stem}.{ext}"
        fig.savefig(p, bbox_inches="tight", dpi=600)
        paths.append(p)
    plt.close(fig)
    print("Saved:", ", ".join(map(str, paths)))

def null_thresholds(values, percentile=NULL_PERCENTILE):
    arr = pd.Series(values).dropna().astype(float)
    neg, pos = arr[arr < 0], arr[arr > 0]
    return (float(neg.quantile(1 - percentile)) if len(neg) else 0.0,
            float(pos.quantile(percentile)) if len(pos) else 0.0)

def bh_fdr(p_values):
    p = np.asarray(p_values, dtype=float)
    if len(p) == 0: return np.array([])
    order = np.argsort(p); ranked = p[order]
    q_ranked = ranked * len(p) / np.arange(1, len(p) + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q = np.empty(len(p)); q[order] = np.clip(q_ranked, 0, 1)
    return q

def score_sign(x, eps=1e-12):
    if pd.isna(x): return "NA"
    if x > eps: return "Positive"
    if x < -eps: return "Negative"
    return "Near-zero"

def format_p(p):
    if pd.isna(p): return "NA"
    if p < 1e-300: return "<1e-300"
    if p < 1e-3: return f"{p:.1e}"
    return f"{p:.3f}"

def short_tf_name(x):
    repl = {
        "EBOX/CAGCTG/CACCTG": "E-box",
        "CREB/ATF/3": "CREB",
        "DRE/3": "DRE",
        "KNI/1": "KNI",
        "MAF/2": "MAF",
        "SREBP/2": "SREBP",
        "HD/16": "HD",
        "OHLER1": "Ohler1",
        "OHLER7": "Ohler7",
    }
    return repl.get(str(x), str(x))

## Null construction and task-specific calibration

The plotting notebook **does not regenerate null samples**. In this notebook, `null_isa.csv` and `null_interaction.csv` are direct input tables loaded with `pd.read_csv` from each completed ISA result folder. The completed CAGE, DEV and HK result folders were generated by the `Ep_ISA_NEW` pipeline (`EpQuickStart.run_isa`), an analysis wrapper adapted from deepISA for Fi-NeMo motif hits. The construction notes below describe the provenance of those precomputed input tables in `Ep_ISA_NEW`; they are not re-executed here.

The null model has two separate levels.

1. **Single-motif null (`null_isa.csv`)**: non-motif pseudo-sites are sampled from the task-specific `non_motif_locs` background with a length distribution matched to the real motif intervals. The same single-ablation ISA calculation used for motif-called intervals is then applied to these pseudo-sites. In `Ep_ISA_NEW`, the default target size is 2,000 null pseudo-sites per task.
2. **Pair-interaction null (`null_interaction.csv`)**: non-motif pseudo-site pairs are sampled from the task-specific non-motif background so that their pair-distance distribution follows the empirical motif-pair distance distribution. The same pair-ablation interaction calculation is then applied to these pseudo-site pairs. In `Ep_ISA_NEW`, the default pair-null settings are `k=9`, target `n=2000`, `receptive_field=255` and `n_bins=20`.

Thresholds are always derived **within each task**. For each task and score type, the negative threshold is the lower tail of negative null values and the positive threshold is the upper tail of positive null values. With `NULL_PERCENTILE = 0.80`, this means the 20th percentile of negative null values and the 80th percentile of positive null values.

This task-specific null construction is why CAGE, DEV and HK should not share or pool null thresholds. The null tables provide empirical background distributions for calibration and filtering; final TF-pair class calls also require grouping, minimum support, statistical testing and FDR correction.

In [ ]:
def build_null_method_summary():
    rows = []
    for t in TASKS:
        single_null = raw[t.label]["null_isa"][isa_col(t)].dropna().astype(float)
        pair_null = raw[t.label]["null_interaction"][interaction_col(t)].dropna().astype(float)
        s_neg, s_pos = null_thresholds(single_null)
        p_neg, p_pos = null_thresholds(pair_null)
        rows.append({
            "task": t.label,
            "track": t.track,
            "single_null_table": str(data_dir(t) / "null_isa.csv"),
            "pair_null_table": str(data_dir(t) / "null_interaction.csv"),
            "single_null_n_loaded": int(len(single_null)),
            "pair_null_n_loaded": int(len(pair_null)),
            "single_null_median": float(single_null.median()),
            "pair_null_median": float(pair_null.median()),
            "single_null_negative_threshold_p20_of_negative_tail": float(s_neg),
            "single_null_positive_threshold_p80_of_positive_tail": float(s_pos),
            "pair_null_negative_threshold_p20_of_negative_tail": float(p_neg),
            "pair_null_positive_threshold_p80_of_positive_tail": float(p_pos),
            "null_percentile_used": NULL_PERCENTILE,
            "single_null_construction": "length-matched non-motif pseudo-sites scored with the same single-ablation ISA calculation",
            "pair_null_construction": "distance-matched non-motif pseudo-site pairs scored with the same pair-ablation interaction calculation",
        })
    return pd.DataFrame(rows)

null_method_summary = build_null_method_summary()
null_method_summary.to_csv(SOURCE_DIR / "null_method_summary.csv", index=False)
panel_context("Methods", "", "Task-specific null construction and thresholds",
              "What null background is used for single-motif ISA and motif-pair interaction in each task?",
              ["Load each task's precomputed null_isa.csv and null_interaction.csv.",
               "Use single-motif null only for single-ablation ISA calibration and support gates.",
               "Use pair-interaction null only for pair-interaction binning, effective interactions and class-level tests.",
               "Derive negative and positive thresholds separately within each task and score type.",
               "Save the audit table so every figure can point to the same null definition."],
              null_method_summary)

## Fig. 1 — full-task scale and task-specific null calibration

Fig. 1 is the global entry point. It uses the full CAGE, DEV and HK ISA output tables and their own pseudo-site null tables. It does not make CAGE-with-DEV or CAGE-with-HK claims.

Important distinction used in all figures:

- **Single-motif ISA records** are motif-called intervals tested one at a time. They do **not** mean that the sequence contains only one motif.
- **Motif-pair interaction records** are pairs of motif-called intervals in the same region, evaluated for non-additive model response after comparing single and double ablations.

In [ ]:
def build_fig1_source():
    rows = []
    for t in TASKS:
        single = raw[t.label]["motif_single_isa"][isa_col(t)].dropna().astype(float)
        single_null = raw[t.label]["null_isa"][isa_col(t)].dropna().astype(float)
        pair = raw[t.label]["motif_combi_isa"][interaction_col(t)].dropna().astype(float)
        pair_null = raw[t.label]["null_interaction"][interaction_col(t)].dropna().astype(float)
        rows.append({"task": t.label, "single_records": len(single), "pair_records": len(pair),
                     "single_real_median": single.median(), "single_null_median": single_null.median(),
                     "pair_real_median": pair.median(), "pair_null_median": pair_null.median(),
                     "single_null_n": len(single_null), "pair_null_n": len(pair_null)})
    return pd.DataFrame(rows)

fig1 = build_fig1_source()
fig1.to_csv(SOURCE_DIR / "fig1_full_task_scale_and_null_source.csv", index=False)
panel_context("Fig. 1", "a-c", "Full-task scale and task-specific null medians",
              "What is the full ISA output scale, and are CAGE/DEV/HK each calibrated against their own null?",
              ["Read complete motif_single_isa, motif_combi_isa, null_isa and null_interaction tables.",
               "Use only the task's own precomputed non-motif pseudo-site null tables.",
               "Count records before any overlap-class filtering.",
               "Compute real and null medians within each task only."], fig1)

In [ ]:
def plot_fig1a(ax):
    x = np.arange(len(TASK_ORDER)); width = 0.34
    sub = fig1.set_index("task").loc[TASK_ORDER]
    ax.bar(x - width/2, sub["single_records"], width, color="#D9D9D9", edgecolor="#4D4D4D", linewidth=0.6, label="Single-motif ISA records")
    ax.bar(x + width/2, sub["pair_records"], width, color="#767676", edgecolor="#4D4D4D", linewidth=0.6, label="Motif-pair interaction records")
    ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(TASK_ORDER)
    ax.set_ylabel("ISA output records (log10 scale)"); ax.set_title("Full-task ISA output"); ax.legend(fontsize=5.8)
    add_panel_label(ax, "a")

def plot_median_lines(ax, real_col, null_col, title, ylabel, label):
    sub = fig1.set_index("task").loc[TASK_ORDER].reset_index()
    x = np.arange(len(TASK_ORDER)); real = sub[real_col].to_numpy(float); null = sub[null_col].to_numpy(float)
    for xi, r, n, task in zip(x, real, null, TASK_ORDER):
        ax.plot([xi, xi], [n, r], color=TASK_COLOR[task], lw=1.25, alpha=0.75)
    ax.plot(x, null, color=NULL_COLOR, marker="o", ms=4.5, lw=1.25, label="Null")
    ax.plot(x, real, color=REAL_COLOR, marker="o", ms=4.5, lw=1.35, label="Motif-called")
    ax.axhline(0, color="#999999", lw=0.7, ls=":")
    ax.set_xticks(x); ax.set_xticklabels(TASK_ORDER); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(fontsize=6)
    add_panel_label(ax, label)

for panel, cols, question in [
    ("a", ["task", "single_records", "pair_records"], "What is the full-task record scale?"),
    ("b", ["task", "single_real_median", "single_null_median", "single_records", "single_null_n"], "Are single-motif ISA medians shifted from task-specific null?"),
    ("c", ["task", "pair_real_median", "pair_null_median", "pair_records", "pair_null_n"], "Are pair-interaction medians shifted from task-specific null?")
]:
    panel_context("Fig. 1", panel, f"Panel {panel} plotting data", question, ["Print exact data used for this panel before drawing."], fig1[cols])

fig, ax = plt.subplots(figsize=(2.35, 2.05))
plot_fig1a(ax)
save_figure(fig, "Fig1a_full_task_record_counts")

fig, ax = plt.subplots(figsize=(2.35, 2.05))
plot_median_lines(ax, "single_real_median", "single_null_median", "Single-motif ISA\n(one motif ablated)", "Median single-motif ISA", "b")
save_figure(fig, "Fig1b_single_motif_median_real_vs_null")

fig, ax = plt.subplots(figsize=(2.35, 2.05))
plot_median_lines(ax, "pair_real_median", "pair_null_median", "Motif-pair interaction\n(two motifs compared)", "Median pair interaction", "c")
save_figure(fig, "Fig1c_pair_interaction_median_real_vs_null")

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.25), gridspec_kw={"width_ratios": [1.05, 1, 1]})
plot_fig1a(axes[0])
plot_median_lines(axes[1], "single_real_median", "single_null_median", "Single-motif ISA\n(one motif ablated)", "Median single-motif ISA", "b")
plot_median_lines(axes[2], "pair_real_median", "pair_null_median", "Motif-pair interaction\n(two motifs compared)", "Median pair interaction", "c")
fig.subplots_adjust(wspace=0.55, bottom=0.22, top=0.82)
save_figure(fig, "Fig1_full_task_scale_and_null")

## Fig. 2 — full real-vs-null distribution checks

Fig. 2 shows whether the median differences in Fig. 1 reflect broader distribution shifts, skew or tails. Each panel uses a task-specific null.

Rows are intentionally separated: the top row is the single-motif ISA score distribution, and the bottom row is the motif-pair interaction score distribution.

In [ ]:
def build_fig2_summary():
    rows = []
    for t in TASKS:
        comps = [("single_motif_isa", raw[t.label]["motif_single_isa"][isa_col(t)], raw[t.label]["null_isa"][isa_col(t)]),
                 ("motif_pair_interaction", raw[t.label]["motif_combi_isa"][interaction_col(t)], raw[t.label]["null_interaction"][interaction_col(t)])]
        for comp, real, null in comps:
            real, null = real.dropna().astype(float), null.dropna().astype(float)
            p = mannwhitneyu(real.to_numpy(), null.to_numpy()).pvalue
            rows.append({"task": t.label, "comparison": comp, "real_n": len(real), "null_n": len(null),
                         "real_median": real.median(), "null_median": null.median(),
                         "real_iqr": real.quantile(.75)-real.quantile(.25), "null_iqr": null.quantile(.75)-null.quantile(.25),
                         "mannwhitney_p": p, "p_label": format_p(p)})
    return pd.DataFrame(rows)

fig2 = build_fig2_summary()
fig2.to_csv(SOURCE_DIR / "fig2_real_vs_null_distribution_summary.csv", index=False)
panel_context("Fig. 2", "all", "Real-vs-null distributions",
              "Do motif-called values separate from pseudo-site nulls across the full distribution?",
              ["Compute n, median, IQR and Mann-Whitney U test for real vs null within each task.",
               "Plot single-motif ISA and pair interaction separately."], fig2)

fig, axes = plt.subplots(2, 3, figsize=(7.2, 3.9))
for col, t in enumerate(TASKS):
    panels = [(0, raw[t.label]["motif_single_isa"][isa_col(t)], raw[t.label]["null_isa"][isa_col(t)], "single_motif_isa", "Single-motif ISA score"),
              (1, raw[t.label]["motif_combi_isa"][interaction_col(t)], raw[t.label]["null_interaction"][interaction_col(t)], "motif_pair_interaction", "Motif-pair interaction score")]
    for row, real, null, comp, xlabel in panels:
        ax = axes[row, col]; real = real.dropna().astype(float); null = null.dropna().astype(float)
        sns.kdeplot(null, ax=ax, color=NULL_COLOR, lw=1.1, label="Null")
        sns.kdeplot(real, ax=ax, color=REAL_COLOR, lw=1.35, label="Motif-called")
        ax.axvline(0, color="#999999", lw=0.7, ls=":")
        s = fig2[(fig2.task == t.label) & (fig2.comparison == comp)].iloc[0]
        ax.text(.03, .92, f"n={int(s.real_n):,}/{int(s.null_n):,}\nmed={s.real_median:.3g}/{s.null_median:.3g}\nP={s.p_label}", transform=ax.transAxes, va="top", fontsize=5.4)
        ax.set_xlabel(xlabel); ax.set_title(t.label if row == 0 else "")
        ax.set_ylabel("Density" if col == 0 else "")
        if row == 0 and col == 0: add_panel_label(ax, "a")
        if row == 1 and col == 0: add_panel_label(ax, "b")
axes[0, 2].legend(fontsize=6)
fig.subplots_adjust(wspace=.48, hspace=.62, bottom=.13, top=.88)
save_figure(fig, "Fig2_full_task_real_vs_null_distributions")

## Fig. 2c — single-motif identity-level ISA heatmap

The distribution panels establish the full-task real-vs-null separation, but they do not show which motif identities contribute to the single-motif ISA layer. Fig. 2c therefore summarizes the median single-motif ISA for each motif identity after subtracting the task-specific single-motif null median. This keeps the panel separate from motif-pair interaction and avoids comparing raw task medians without null calibration.

In [ ]:
def build_single_motif_identity_heatmap_source():
    rows = []
    for t in TASKS:
        single = raw[t.label]["motif_single_isa"].copy()
        val = isa_col(t)
        null_median = raw[t.label]["null_isa"][val].dropna().astype(float).median()
        null_neg, null_pos = null_thresholds(raw[t.label]["null_isa"][val])
        for tf, g in single.groupby("tf"):
            vals = g[val].dropna().astype(float)
            if len(vals) == 0:
                continue
            rows.append({
                "task": t.label,
                "tf": tf,
                "motif": short_tf_name(tf),
                "n_single_motif_records": len(vals),
                "median_single_isa": vals.median(),
                "task_null_median": null_median,
                "null_adjusted_median_single_isa": vals.median() - null_median,
                "fraction_above_positive_null_threshold": float((vals > null_pos).mean()),
                "single_null_negative_threshold": null_neg,
                "single_null_positive_threshold": null_pos,
            })
    return pd.DataFrame(rows)

fig2c_source = build_single_motif_identity_heatmap_source()
fig2c_source.to_csv(SOURCE_DIR / "fig2c_single_motif_identity_null_adjusted_source.csv", index=False)
panel_context("Fig. 2", "c", "Single-motif identity-level null-adjusted ISA",
              "Which motif identities drive the single-motif ISA layer after task-specific null calibration?",
              ["Group motif_single_isa rows by task and motif identity.",
               "Compute the median single-motif ISA per motif identity.",
               "Subtract that task's own single-motif null median before cross-task display.",
               "Report n and the fraction above the task-specific positive single-motif null threshold in source data."],
              fig2c_source.sort_values(["task", "null_adjusted_median_single_isa"], ascending=[True, False]))

def plot_single_identity_heatmap(df):
    motifs = sorted(df["tf"].unique(), key=lambda x: short_tf_name(x))
    labels = [short_tf_name(m) for m in motifs]
    mat = pd.DataFrame(np.nan, index=motifs, columns=TASK_ORDER)
    nmat = pd.DataFrame(0, index=motifs, columns=TASK_ORDER, dtype=int)
    for _, r in df.iterrows():
        mat.loc[r.tf, r.task] = r.null_adjusted_median_single_isa
        nmat.loc[r.tf, r.task] = int(r.n_single_motif_records)
    vmax = float(np.nanpercentile(mat.to_numpy(float), 98))
    fig, ax = plt.subplots(figsize=(2.55, 2.95))
    cmap = mpl.colormaps["YlOrBr"].copy()
    cmap.set_bad("#EDEDED")
    im = ax.imshow(mat.to_numpy(float), cmap=cmap, vmin=0, vmax=max(vmax, 0.1), aspect="auto")
    ax.set_xticks(np.arange(len(TASK_ORDER))); ax.set_xticklabels(TASK_ORDER, fontsize=6)
    for tick, task in zip(ax.get_xticklabels(), TASK_ORDER):
        tick.set_color(TASK_COLOR[task])
    ax.set_yticks(np.arange(len(motifs))); ax.set_yticklabels(labels, fontsize=5.8)
    ax.set_title("Single-motif identity response", fontsize=7.2, pad=7)
    ax.set_xlabel("Task")
    ax.set_ylabel("Motif identity")
    for i in range(len(motifs)):
        for j, task in enumerate(TASK_ORDER):
            n = nmat.iloc[i, j]
            if n > 0:
                ax.text(j, i, f"{n}", ha="center", va="center", fontsize=4.6,
                        color="white" if mat.iloc[i, j] > vmax * 0.58 else "#272727")
    for x in np.arange(len(TASK_ORDER) + 1) - 0.5:
        ax.axvline(x, color="white", lw=0.45)
    for y in np.arange(len(motifs) + 1) - 0.5:
        ax.axhline(y, color="white", lw=0.45)
    for spine in ax.spines.values():
        spine.set_visible(False)
    add_panel_label(ax, "c", x=-0.24, y=1.03)
    cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.035)
    cbar.set_label("Median single ISA - task null median", fontsize=5.8)
    cbar.ax.tick_params(labelsize=5.4, length=2)
    fig.subplots_adjust(left=.28, right=.86, bottom=.14, top=.86)
    return fig

fig = plot_single_identity_heatmap(fig2c_source)
save_figure(fig, "Fig2c_single_motif_identity_null_adjusted_heatmap")

## Fig. 3 — from motif-pair instances to direction-supported TF-pair non-additivity

Fig. 3 is the main bridge from all motif-pair interaction instances to direction-supported TF-pair classes. It is intentionally multi-panel because the class-level result is not a direct deduplication of raw rows. `Near-null` exists for raw instances; direction-supported TF-pair classes may have no near-zero class after FDR and effective-count filtering.

This figure contains only motif-pair-derived quantities. It should not be read as a single-motif ISA analysis.

Before TF-pair class aggregation, the figure first asks which motif-pair instances have both single motifs above the task-specific single-motif null threshold, which have pair interaction outside the task-specific pair-interaction null threshold, and which satisfy both. It then shows the class-level funnel and the final net non-additivity evidence.

In [ ]:
def canonicalize_pairs(df):
    out = df.copy()
    a = np.minimum(out.tf1.astype(str), out.tf2.astype(str)); b = np.maximum(out.tf1.astype(str), out.tf2.astype(str))
    out["tf1"], out["tf2"], out["tf_pair"] = a, b, a + "|" + b
    return out.drop_duplicates()

def build_pair_instance_intersection_audit():
    rows = []
    detail_frames = []
    for t in TASKS:
        combi = canonicalize_pairs(raw[t.label]["motif_combi_isa"])
        null_isa = raw[t.label]["null_isa"][isa_col(t)].dropna().astype(float)
        null_i = raw[t.label]["null_interaction"][interaction_col(t)].dropna().astype(float)
        single_thr = null_thresholds(null_isa)[1]
        neg_i_thr, pos_i_thr = null_thresholds(null_i)
        isa1, isa2, inter = f"isa1_t{t.track}", f"isa2_t{t.track}", interaction_col(t)
        valid = combi[[isa1, isa2, inter]].notna().all(axis=1)
        combi = combi.loc[valid].copy()
        combi["passes_single_motif_null_support"] = (combi[isa1] >= single_thr) & (combi[isa2] >= single_thr)
        combi["passes_pair_interaction_null"] = (combi[inter] < neg_i_thr) | (combi[inter] > pos_i_thr)
        combi["passes_both_single_and_pair_null"] = combi["passes_single_motif_null_support"] & combi["passes_pair_interaction_null"]
        rows.append({
            "task": t.label,
            "all_valid_pair_instances": len(combi),
            "single_motif_null_supported_instances": int(combi["passes_single_motif_null_support"].sum()),
            "pair_interaction_null_exceeding_instances": int(combi["passes_pair_interaction_null"].sum()),
            "intersection_single_supported_and_pair_null_exceeding": int(combi["passes_both_single_and_pair_null"].sum()),
            "single_only_not_pair_null": int((combi["passes_single_motif_null_support"] & ~combi["passes_pair_interaction_null"]).sum()),
            "pair_null_only_not_single_supported": int((~combi["passes_single_motif_null_support"] & combi["passes_pair_interaction_null"]).sum()),
            "neither_gate": int((~combi["passes_single_motif_null_support"] & ~combi["passes_pair_interaction_null"]).sum()),
            "single_motif_positive_null_threshold": float(single_thr),
            "pair_interaction_negative_null_threshold": float(neg_i_thr),
            "pair_interaction_positive_null_threshold": float(pos_i_thr),
        })
        detail_frames.append(combi[["region", "tf1", "tf2", "tf_pair", isa1, isa2, inter,
                                   "passes_single_motif_null_support", "passes_pair_interaction_null",
                                   "passes_both_single_and_pair_null"]].assign(task=t.label))
    summary = pd.DataFrame(rows)
    details = pd.concat(detail_frames, ignore_index=True)
    return summary, details

pair_intersection, pair_intersection_details = build_pair_instance_intersection_audit()
pair_intersection.to_csv(SOURCE_DIR / "fig3_bridge_pair_instance_single_and_pair_null_intersection.csv", index=False)
pair_intersection_details.to_csv(SOURCE_DIR / "fig3_bridge_pair_instance_intersection_details.csv", index=False)
panel_context("Fig. 3 bridge", "", "Pair-instance intersection between single-motif support and pair-interaction null evidence",
              "From all motif-pair interaction instances, which rows satisfy single-motif null support, pair-interaction null evidence, or both?",
              ["Start from all valid motif-pair interaction rows after TF-pair canonicalization.",
               "Mark rows where both single motifs have ISA values above the task-specific positive single-motif null threshold.",
               "Mark rows where pair interaction falls outside task-specific negative/positive pair-interaction null thresholds.",
               "Report the intersection before any TF-pair class aggregation, conditional gate or FDR step."],
              pair_intersection)

def plot_pair_intersection_bridge(ax, summary, panel_label="a"):
    sub = summary.set_index("task").loc[TASK_ORDER].reset_index()
    parts = [
        ("neither_gate", "Neither gate", "#E0E0E0"),
        ("single_only_not_pair_null", "Single-ISA gate only", "#9E9E9E"),
        ("pair_null_only_not_single_supported", "Pair gate only", "#7EA6C8"),
        ("intersection_single_supported_and_pair_null_exceeding", "Both gates", "#272727"),
    ]
    y = np.arange(len(sub))
    left = np.zeros(len(sub))
    total = sub["all_valid_pair_instances"].to_numpy(float)
    for col, label, color in parts:
        vals = sub[col].to_numpy(float)
        frac = vals / total * 100
        ax.barh(y, frac, left=left, color=color, edgecolor="white", linewidth=0.6, height=0.62, label=label)
        for yi, lft, width, val in zip(y, left, frac, vals):
            if width >= 9:
                ax.text(lft + width / 2, yi, f"{width:.0f}%", ha="center", va="center", fontsize=5.6,
                        color="white" if color == "#272727" else "#272727")
            elif label == "Both gates":
                ax.text(lft + width + 1.0, yi, f"{int(val):,}", ha="left", va="center", fontsize=5.3, color="#272727")
        left += frac
    for yi, task, n in zip(y, sub["task"], sub["all_valid_pair_instances"]):
        ax.text(101.5, yi, f"n={int(n):,}", ha="left", va="center", fontsize=5.8, color=TASK_COLOR[task])
    ax.set_yticks(y); ax.set_yticklabels(sub["task"])
    ax.invert_yaxis()
    ax.set_xlim(0, 112)
    ax.set_xlabel("Valid motif-pair instances (%)")
    ax.set_title("Instance gate intersection", loc="left", fontsize=7.2, pad=8)
    ax.legend(fontsize=5.4, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.19),
              handlelength=1.0, columnspacing=0.8, borderaxespad=0)
    add_panel_label(ax, panel_label)

fig, ax = plt.subplots(figsize=(5.0, 2.25))
plot_pair_intersection_bridge(ax, pair_intersection)
fig.subplots_adjust(left=.14, right=.88, bottom=.24, top=.72)
save_figure(fig, "Fig3_bridge_pair_instance_single_and_pair_null_intersection")

def build_fig3():
    inst_rows, cascade_rows, class_rows = [], [], []
    for t in TASKS:
        combi = canonicalize_pairs(raw[t.label]["motif_combi_isa"])
        null_isa = raw[t.label]["null_isa"][isa_col(t)].dropna().astype(float)
        null_i = raw[t.label]["null_interaction"][interaction_col(t)].dropna().astype(float)
        isa_thr = null_thresholds(null_isa)[1]; neg_thr, pos_thr = null_thresholds(null_i)
        inter = interaction_col(t); isa1, isa2, both = f"isa1_t{t.track}", f"isa2_t{t.track}", f"isa_both_t{t.track}"
        real_i = combi[inter].dropna().astype(float)
        bins = np.where(real_i < neg_thr, "Negative", np.where(real_i > pos_thr, "Positive", "Near-null"))
        for cls, n in pd.Series(bins).value_counts().reindex(["Negative", "Near-null", "Positive"], fill_value=0).items():
            inst_rows.append({"task": t.label, "sign_class": cls, "n_instances": int(n), "fraction": n / len(real_i),
                              "negative_null_threshold": neg_thr, "positive_null_threshold": pos_thr, "real_n": len(real_i), "null_n": len(null_i)})
        combi["isa1_wo2"] = combi[both] - combi[isa1]; combi["isa2_wo1"] = combi[both] - combi[isa2]
        pass_single = (combi[isa1] >= isa_thr) & (combi[isa2] >= isa_thr)
        pass_cond = (combi["isa1_wo2"] >= 0) & (combi["isa2_wo1"] >= 0)
        eligible = combi[pass_single & pass_cond].copy()
        cascade_rows.append({"task": t.label, "raw_pair_instances": len(combi), "instances_after_single_isa_support": int(pass_single.sum()),
                             "instances_after_current_eligibility_gate": len(eligible), "unique_tf_pairs_raw": combi.tf_pair.nunique(),
                             "unique_tf_pairs_eligible": eligible.tf_pair.nunique(), "null_isa_positive_threshold": isa_thr,
                             "null_interaction_negative_threshold": neg_thr, "null_interaction_positive_threshold": pos_thr})
        for pair, g in eligible.groupby("tf_pair"):
            vals = g[inter].dropna().astype(float).to_numpy()
            if len(vals) < MIN_EFFECTIVE_COUNT: continue
            eff = vals[(vals < neg_thr) | (vals > pos_thr)]
            p = mannwhitneyu(vals, null_i.to_numpy()).pvalue
            c_all = vals.sum() / np.abs(vals).sum() if np.abs(vals).sum() else np.nan
            c_eff = eff.sum() / np.abs(eff).sum() if len(eff) and np.abs(eff).sum() else np.nan
            class_rows.append({"task": t.label, "tf_pair": pair, "n_eligible_instances": len(vals), "n_effective_instances": len(eff),
                               "C_all": c_all, "C_effective": c_eff, "mw_p": p, "median_distance": g.distance.median(),
                               "null_interaction_negative_threshold": neg_thr, "null_interaction_positive_threshold": pos_thr})
    inst, cascade, classes = pd.DataFrame(inst_rows), pd.DataFrame(cascade_rows), pd.DataFrame(class_rows)
    if len(classes):
        classes["mw_q"] = np.nan
        for task, idx in classes.groupby("task").groups.items():
            classes.loc[idx, "mw_q"] = bh_fdr(classes.loc[idx, "mw_p"])
        classes["passes_q"] = classes.mw_q < Q_VALUE_THRESHOLD
        classes["passes_effective_count"] = classes.n_effective_instances >= MIN_EFFECTIVE_COUNT
        classes["direction_supported"] = classes.passes_q & classes.passes_effective_count
        classes["direction_class"] = classes.C_effective.map(score_sign)
        classes.loc[~classes.direction_supported, "direction_class"] = "Not direction-supported"
    return inst, cascade, classes

fig3_inst, fig3_cascade, fig3_classes = build_fig3()
fig3_inst.to_csv(SOURCE_DIR / "fig3a_all_pair_instances_null_binned.csv", index=False)
fig3_cascade.to_csv(SOURCE_DIR / "fig3_filtering_cascade.csv", index=False)
fig3_classes.to_csv(SOURCE_DIR / "fig3_tf_pair_class_table.csv", index=False)

class_funnel_rows = []
for task in TASK_ORDER:
    c = fig3_cascade.set_index("task").loc[task]
    sub = fig3_classes[fig3_classes.task == task]
    class_funnel_rows.append({
        "task": task,
        "raw_tf_pair_classes": int(c["unique_tf_pairs_raw"]),
        "eligible_tf_pair_classes": int(c["unique_tf_pairs_eligible"]),
        "tested_classes_n_ge_10": int(len(sub)),
        "q_supported_classes": int(sub["passes_q"].sum()),
        "direction_supported_classes": int(sub["direction_supported"].sum()),
        "negative_direction_supported": int(((sub["direction_supported"]) & (sub["direction_class"] == "Negative")).sum()),
        "positive_direction_supported": int(((sub["direction_supported"]) & (sub["direction_class"] == "Positive")).sum()),
    })
fig3_class_funnel = pd.DataFrame(class_funnel_rows)
fig3_class_funnel.to_csv(SOURCE_DIR / "fig3_bridge_class_level_funnel.csv", index=False)

fdr_sensitivity_rows = []
for threshold in [0.10, 0.05]:
    for task in TASK_ORDER:
        sub = fig3_classes[(fig3_classes.task == task) & (fig3_classes["passes_effective_count"]) & (fig3_classes["mw_q"] < threshold)].copy()
        sub["threshold_direction_class"] = sub["C_effective"].map(score_sign)
        fdr_sensitivity_rows.append({
            "task": task,
            "q_threshold": threshold,
            "direction_supported_classes": int(len(sub)),
            "negative_classes": int((sub["threshold_direction_class"] == "Negative").sum()),
            "positive_classes": int((sub["threshold_direction_class"] == "Positive").sum()),
            "near_zero_classes": int((sub["threshold_direction_class"] == "Near-zero").sum()),
        })
fig3_fdr_sensitivity = pd.DataFrame(fdr_sensitivity_rows)
fig3_fdr_sensitivity.to_csv(SOURCE_DIR / "fig3_bridge_fdr_threshold_sensitivity.csv", index=False)

panel_context("Fig. 3 bridge", "b", "Class-level funnel after instance-level gates",
              "After the instance-level intersection, how do motif-pair instances become the final direction-supported TF-pair classes?",
              ["Canonicalize and group instances into TF-pair classes.",
               "Apply current eligibility gates and require at least 10 eligible instances for class testing.",
               "Test each class against the task-specific pair-interaction null distribution.",
               "Apply task-wise BH FDR and require at least 10 effective null-exceeding interactions.",
               "The final class counts are therefore not a direct deduplication of the black instance-level segment."],
              fig3_class_funnel)
panel_context("Fig. 3 bridge", "c", "FDR threshold sensitivity",
              "Are direction-supported TF-pair class counts qualitatively stable if q<0.05 is used instead of the discovery-level q<0.10 threshold?",
              ["Recompute direction-supported TF-pair class counts using the same effective-count rule.",
               "Compare the discovery-level q<0.10 threshold with a stricter q<0.05 threshold.",
               "Report total, negative and positive direction-supported classes for each task."],
              fig3_fdr_sensitivity)

def plot_class_funnel(ax, funnel, panel_label="b"):
    steps = [
        ("raw_tf_pair_classes", "Raw\nTF-pair\nclasses"),
        ("eligible_tf_pair_classes", "Eligibility\ngate"),
        ("tested_classes_n_ge_10", "Eligible\nn>=10"),
        ("q_supported_classes", "FDR\nq<0.10"),
        ("direction_supported_classes", "Final direction-\nsupported"),
    ]
    mat = funnel.set_index("task").loc[TASK_ORDER, [s[0] for s in steps]].to_numpy(float)
    im = ax.imshow(mat, cmap="Greys", aspect="auto", vmin=0, vmax=max(60, float(np.nanmax(mat))))
    for i, task in enumerate(TASK_ORDER):
        for j, val in enumerate(mat[i]):
            color = "white" if val > np.nanmax(mat) * 0.62 else "#272727"
            ax.text(j, i, f"{int(val)}", ha="center", va="center", fontsize=6.0, color=color)
    ax.set_xticks(np.arange(len(steps))); ax.set_xticklabels([s[1] for s in steps], fontsize=5.8)
    ax.set_yticks(np.arange(len(TASK_ORDER))); ax.set_yticklabels(TASK_ORDER)
    for tick, task in zip(ax.get_yticklabels(), TASK_ORDER):
        tick.set_color(TASK_COLOR[task])
    ax.set_title("Class-level filtering", loc="left", fontsize=7.2, pad=8)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    add_panel_label(ax, panel_label)

def plot_fdr_sensitivity(ax, sensitivity, panel_label="f"):
    thresholds = [0.10, 0.05]
    mat = []
    ann = []
    for task in TASK_ORDER:
        row = []
        ann_row = []
        for threshold in thresholds:
            s = sensitivity[(sensitivity.task == task) & (sensitivity.q_threshold == threshold)].iloc[0]
            row.append(float(s["direction_supported_classes"]))
            ann_row.append(f"{int(s['direction_supported_classes'])}\nN{int(s['negative_classes'])}/P{int(s['positive_classes'])}")
        mat.append(row)
        ann.append(ann_row)
    mat = np.asarray(mat, dtype=float)
    im = ax.imshow(mat, cmap="Greys", aspect="auto", vmin=0, vmax=max(30, float(np.nanmax(mat))))
    for i, task in enumerate(TASK_ORDER):
        for j, threshold in enumerate(thresholds):
            val = mat[i, j]
            color = "white" if val > np.nanmax(mat) * 0.62 else "#272727"
            ax.text(j, i, ann[i][j], ha="center", va="center", fontsize=5.8, color=color, linespacing=0.95)
    ax.set_xticks(np.arange(len(thresholds))); ax.set_xticklabels(["Discovery\nq<0.10", "Stringent\nq<0.05"])
    ax.set_yticks(np.arange(len(TASK_ORDER))); ax.set_yticklabels(TASK_ORDER)
    for tick, task in zip(ax.get_yticklabels(), TASK_ORDER):
        tick.set_color(TASK_COLOR[task])
    ax.set_title("FDR sensitivity", loc="left", fontsize=7.2, pad=8)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    add_panel_label(ax, panel_label)

fig, axes = plt.subplots(1, 3, figsize=(8.8, 2.55), gridspec_kw={"width_ratios": [1.45, 1.0, 0.62]})
plot_pair_intersection_bridge(axes[0], pair_intersection, panel_label="a")
plot_class_funnel(axes[1], fig3_class_funnel)
plot_fdr_sensitivity(axes[2], fig3_fdr_sensitivity)
fig.subplots_adjust(left=.085, right=.985, bottom=.25, top=.70, wspace=.48)
save_figure(fig, "Fig3ab_instance_to_tf_pair_class_filtering_bridge")

panel_context("Fig. 3", "a", "Raw pair instances binned by null thresholds",
              "How are all raw motif-pair instances distributed relative to task-specific null thresholds?",
              ["Use every motif_combi_isa row.", "Classify by each task's null_interaction thresholds."], fig3_inst)
panel_context("Fig. 3", "filter", "TF-pair class filtering cascade",
              "Where do class counts change before direction-supported TF-pair classes are reported?",
              ["Canonicalize TF pairs.", "Apply single-ISA support and current conditional nonnegative gate.", "Group by TF-pair, test against task-specific null, then apply BH FDR and effective-count filters."], fig3_cascade)
panel_context("Fig. 3", "b-c", "Direction-supported TF-pair classes",
              "Which TF-pair classes retain a directional net non-additivity score after FDR and effective-count filtering?",
              ["Compute C_all and C_effective.", "Report sign only for q < 0.10 and n_effective >= 10."],
              fig3_classes.sort_values(["task", "direction_supported", "mw_q"], ascending=[True, False, True]))

In [ ]:
def plot_fig3a(ax, panel_label="c"):
    pivot = fig3_inst.pivot_table(index="task", columns="sign_class", values="n_instances", aggfunc="sum").reindex(TASK_ORDER).fillna(0)
    for c in ["Negative", "Near-null", "Positive"]:
        if c not in pivot: pivot[c] = 0
    bottom = np.zeros(len(TASK_ORDER)); x = np.arange(len(TASK_ORDER))
    for cls in ["Negative", "Near-null", "Positive"]:
        ax.bar(x, pivot[cls], bottom=bottom, color=SIGN_COLORS[cls], width=.62, label=cls); bottom += pivot[cls].to_numpy()
    ax.set_xticks(x); ax.set_xticklabels(TASK_ORDER); ax.set_ylabel("Raw motif-pair interaction instances"); ax.set_title("All motif-pair instances"); ax.legend(fontsize=6); add_panel_label(ax, panel_label)

def plot_fig3b(ax, panel_label="d"):
    sub = fig3_classes[fig3_classes.direction_supported]
    for task in TASK_ORDER:
        vals = sub.loc[sub.task == task, "C_effective"].dropna().sort_values().to_numpy()
        if len(vals):
            ax.step(vals, np.arange(1, len(vals)+1)/len(vals), where="post", color=TASK_COLOR[task], lw=1.4, label=f"{task} (n={len(vals)})")
    ax.axvline(0, color="#999999", lw=.7, ls=":"); ax.set_xlabel("TF-pair net non-additivity score"); ax.set_ylabel("ECDF"); ax.set_title("Direction-supported TF-pair classes"); ax.legend(fontsize=6); add_panel_label(ax, panel_label)

def plot_fig3c(ax, panel_label="e"):
    counts = fig3_classes[fig3_classes.direction_supported].groupby(["task", "direction_class"]).size().reset_index(name="n")
    print("Fig. 3c direction-supported class counts:"); display(counts)
    counts.to_csv(SOURCE_DIR / "fig3c_direction_supported_class_counts.csv", index=False)
    pivot = counts.pivot_table(index="task", columns="direction_class", values="n", fill_value=0).reindex(TASK_ORDER).fillna(0)
    present_classes = [c for c in ["Negative", "Near-zero", "Positive"] if c in pivot.columns and pivot[c].sum() > 0]
    for c in present_classes:
        if c not in pivot: pivot[c] = 0
    bottom = np.zeros(len(TASK_ORDER)); x = np.arange(len(TASK_ORDER))
    for cls in present_classes:
        ax.bar(x, pivot[cls], bottom=bottom, color=SIGN_COLORS[cls], width=.62, label=cls); bottom += pivot[cls].to_numpy()
    ax.set_xticks(x); ax.set_xticklabels(TASK_ORDER); ax.set_ylabel("Direction-supported TF-pair classes"); ax.set_title("TF-pair class signs")
    if present_classes:
        ax.legend(fontsize=6)
    add_panel_label(ax, panel_label)

fig, ax = plt.subplots(figsize=(2.3, 2.1))
plot_fig3a(ax, panel_label="c")
save_figure(fig, "Fig3a_all_raw_pair_instances_null_binned")

fig, ax = plt.subplots(figsize=(2.5, 2.1))
plot_fig3b(ax, panel_label="d")
save_figure(fig, "Fig3b_tf_pair_class_score_ecdf")

fig, ax = plt.subplots(figsize=(2.2, 2.1))
plot_fig3c(ax, panel_label="e")
save_figure(fig, "Fig3c_direction_supported_class_sign_counts")

fig, axes = plt.subplots(
    2, 3, figsize=(8.9, 4.7),
    gridspec_kw={"width_ratios": [1.35, 1.0, 0.78], "height_ratios": [1.0, 1.0]}
)
plot_pair_intersection_bridge(axes[0, 0], pair_intersection, panel_label="a")
plot_class_funnel(axes[0, 1], fig3_class_funnel, panel_label="b")
plot_fig3a(axes[0, 2], panel_label="c")
plot_fig3b(axes[1, 0], panel_label="d")
plot_fig3c(axes[1, 1], panel_label="e")
plot_fdr_sensitivity(axes[1, 2], fig3_fdr_sensitivity, panel_label="f")
fig.subplots_adjust(left=.08, right=.985, bottom=.12, top=.86, wspace=.52, hspace=.72)
save_figure(fig, "Fig3_instance_to_tf_pair_nonadditivity")

## Fig. 3g — motif-pair-resolution heatmap

The previous Fig. 3 panels summarize the filtering chain and class-level sign counts. This panel goes one level deeper: it shows which specific TF-pair classes drive the task-level non-additivity pattern. Columns are selected top direction-supported TF-pair classes, the upper heatmap shows their task-wise `C_effective`, and the lower strip marks whether each pair is direction-supported in CAGE, DEV and/or HK.

In [ ]:
TOP_N_PER_TASK = 7

def short_pair_label(pair):
    a, b = str(pair).split("|")
    return f"{short_tf_name(a)}|{short_tf_name(b)}"

def build_top_pair_resolution():
    ds = fig3_classes[fig3_classes.direction_supported].copy()
    ds["abs_C_effective"] = ds["C_effective"].abs()
    selected = []
    for task in TASK_ORDER:
        top = ds[ds.task == task].sort_values(["abs_C_effective", "n_effective_instances"], ascending=False).head(TOP_N_PER_TASK)
        selected.extend(top.tf_pair.tolist())
    selected = sorted(set(selected))
    meta_rows = []
    for pair in selected:
        sub = ds[ds.tf_pair == pair]
        membership = [task for task in TASK_ORDER if ((sub.task == task).any())]
        max_abs = sub.C_effective.abs().max()
        total_eff = int(sub.n_effective_instances.sum())
        meta_rows.append({
            "tf_pair": pair,
            "short_label": short_pair_label(pair),
            "membership_pattern": "+".join(membership),
            "membership_n": len(membership),
            "max_abs_C_effective": float(max_abs),
            "total_effective_instances": total_eff,
        })
    meta = pd.DataFrame(meta_rows)
    pattern_rank = {
        "CAGE": 0, "CAGE+DEV": 1, "CAGE+HK": 2, "CAGE+DEV+HK": 3,
        "DEV": 4, "DEV+HK": 5, "HK": 6
    }
    meta["pattern_rank"] = meta["membership_pattern"].map(pattern_rank).fillna(99)
    meta = meta.sort_values(["pattern_rank", "max_abs_C_effective", "total_effective_instances"], ascending=[True, False, False])
    order = meta.tf_pair.tolist()
    value = ds.pivot_table(index="task", columns="tf_pair", values="C_effective", aggfunc="first").reindex(TASK_ORDER, columns=order)
    support = ds.assign(supported=1).pivot_table(index="task", columns="tf_pair", values="supported", aggfunc="max").reindex(TASK_ORDER, columns=order).fillna(0).astype(int)
    long_rows = []
    for task in TASK_ORDER:
        for pair in order:
            row = ds[(ds.task == task) & (ds.tf_pair == pair)]
            if len(row):
                r = row.iloc[0].to_dict()
                long_rows.append({
                    "task": task, "tf_pair": pair, "short_label": short_pair_label(pair),
                    "C_effective": r["C_effective"], "n_effective_instances": r["n_effective_instances"],
                    "mw_q": r["mw_q"], "direction_class": r["direction_class"],
                    "direction_supported": True,
                })
            else:
                long_rows.append({
                    "task": task, "tf_pair": pair, "short_label": short_pair_label(pair),
                    "C_effective": np.nan, "n_effective_instances": 0,
                    "mw_q": np.nan, "direction_class": "Not direction-supported",
                    "direction_supported": False,
                })
    long = pd.DataFrame(long_rows).merge(meta.drop(columns=["pattern_rank"]), on=["tf_pair", "short_label"], how="left")
    return meta.drop(columns=["pattern_rank"]), value, support, long

fig3g_meta, fig3g_value, fig3g_support, fig3g_long = build_top_pair_resolution()
fig3g_meta.to_csv(SOURCE_DIR / "fig3g_top_tf_pair_resolution_selected_pairs.csv", index=False)
fig3g_long.to_csv(SOURCE_DIR / "fig3g_top_tf_pair_resolution_long_source.csv", index=False)

panel_context("Fig. 3", "g", "Top TF-pair classes at motif-pair resolution",
              "Which specific TF-pair classes underlie the task-level non-additivity patterns?",
              ["Start from direction-supported TF-pair classes from Fig. 3.",
               f"Select the top {TOP_N_PER_TASK} classes per task by absolute C_effective, then take the union.",
               "Plot C_effective for selected pairs across CAGE, DEV and HK.",
               "Use the lower membership strip to show whether each pair is direction-supported in each task."],
              fig3g_long)

def plot_pair_resolution_heatmap(value, support, meta):
    labels = [short_pair_label(p) for p in value.columns]
    n = len(labels)
    fig = plt.figure(figsize=(max(7.2, n * 0.38), 4.05))
    gs = fig.add_gridspec(3, 2, height_ratios=[1.0, 0.42, 0.17], width_ratios=[1, 0.035],
                          hspace=0.10, wspace=0.04)
    ax_h = fig.add_subplot(gs[0, 0])
    ax_c = fig.add_subplot(gs[0, 1])
    ax_s = fig.add_subplot(gs[1, 0])
    ax_n = fig.add_subplot(gs[2, 0])

    cmap = mpl.colormaps["RdBu_r"].copy()
    cmap.set_bad("#EDEDED")
    im = ax_h.imshow(value.to_numpy(float), aspect="auto", cmap=cmap, vmin=-1, vmax=1)
    cbar = fig.colorbar(im, cax=ax_c)
    cbar.set_label("C_effective", fontsize=6)
    cbar.ax.tick_params(labelsize=5.5, length=2)
    ax_h.set_yticks(np.arange(len(TASK_ORDER))); ax_h.set_yticklabels(TASK_ORDER)
    for tick, task in zip(ax_h.get_yticklabels(), TASK_ORDER):
        tick.set_color(TASK_COLOR[task])
    ax_h.set_xticks(np.arange(n)); ax_h.set_xticklabels([])
    ax_h.tick_params(axis="x", bottom=False, labelbottom=False)
    ax_h.set_title("Top direction-supported TF-pair classes", loc="left", fontsize=7.5, pad=7)
    ax_h.tick_params(length=0)
    for spine in ax_h.spines.values():
        spine.set_visible(False)
    for x in np.arange(n + 1) - 0.5:
        ax_h.axvline(x, color="white", lw=0.35)
    for y in np.arange(len(TASK_ORDER) + 1) - 0.5:
        ax_h.axhline(y, color="white", lw=0.55)

    ax_s.set_xlim(-0.5, n - 0.5)
    ax_s.set_ylim(len(TASK_ORDER) - 0.5, -0.5)
    for i, task in enumerate(TASK_ORDER):
        for j, pair in enumerate(value.columns):
            supported = bool(support.loc[task, pair])
            color = TASK_COLOR[task] if supported else "#F0F0F0"
            ax_s.add_patch(mpl.patches.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor=color,
                                                 edgecolor="white", linewidth=0.45))
    ax_s.set_yticks(np.arange(len(TASK_ORDER))); ax_s.set_yticklabels(TASK_ORDER)
    for tick, task in zip(ax_s.get_yticklabels(), TASK_ORDER):
        tick.set_color(TASK_COLOR[task])
    ax_s.set_xticks(np.arange(n)); ax_s.set_xticklabels([])
    ax_s.tick_params(length=0, axis="x", labelbottom=False, labeltop=False)
    ax_s.set_ylabel("Supported", fontsize=6)
    for spine in ax_s.spines.values():
        spine.set_visible(False)

    n_map = meta.set_index("tf_pair")["total_effective_instances"].reindex(value.columns).astype(float)
    log_n = np.log10(n_map.to_numpy() + 1)
    ax_n.imshow(log_n.reshape(1, -1), aspect="auto", cmap="Greys", vmin=1, vmax=max(3, float(np.nanmax(log_n))))
    ax_n.set_yticks([0]); ax_n.set_yticklabels(["n_eff"], fontsize=6)
    ax_n.set_xticks(np.arange(n)); ax_n.set_xticklabels(labels, rotation=62, ha="right", fontsize=6.0)
    ax_n.tick_params(length=0, axis="x", pad=2)
    for j, val in enumerate(n_map.to_numpy()):
        if val <= 15:
            ax_n.text(j, 0, f"{int(val)}", ha="center", va="center", fontsize=5.2, color="#222222")
    for x in np.arange(n + 1) - 0.5:
        ax_n.axvline(x, color="white", lw=0.45)
    for spine in ax_n.spines.values():
        spine.set_visible(False)

    add_panel_label(ax_h, "g", x=-0.065, y=1.08)
    fig.subplots_adjust(left=0.08, right=0.95, bottom=0.30, top=0.88)
    return fig

fig = plot_pair_resolution_heatmap(fig3g_value, fig3g_support, fig3g_meta)
save_figure(fig, "Fig3g_top_tf_pair_resolution_heatmap")

## Fig. 3h — complete TF-pair non-additivity matrix

Fig. 3g intentionally shows selected top TF-pair classes. Fig. 3h audits that selection against the complete motif-pair space. Each task is shown as a symmetric TF-pair matrix. Colored cells are direction-supported TF-pair classes; grey cells are not direction-supported under the current filters.

In [ ]:
def build_full_tf_pair_matrix_source():
    motifs = sorted(set(sum([str(p).split("|") for p in fig3_classes.tf_pair.dropna().unique()], [])))
    rows = []
    for task in TASK_ORDER:
        sub = fig3_classes[fig3_classes.task == task].copy()
        for _, r in sub.iterrows():
            a, b = str(r.tf_pair).split("|")
            rows.append({
                "task": task, "tf1": a, "tf2": b, "tf_pair": r.tf_pair,
                "C_effective": r.C_effective,
                "n_effective_instances": r.n_effective_instances,
                "mw_q": r.mw_q,
                "direction_class": r.direction_class,
                "direction_supported": bool(r.direction_supported),
            })
    source = pd.DataFrame(rows)
    return motifs, source

fig3h_motifs, fig3h_source = build_full_tf_pair_matrix_source()
fig3h_source.to_csv(SOURCE_DIR / "fig3h_complete_tf_pair_nonadditivity_matrix_source.csv", index=False)

panel_context("Fig. 3", "h", "Complete TF-pair non-additivity matrix",
              "Do the task-specific direction patterns in Fig. 3g hold across the full TF-pair class space?",
              ["Use all tested TF-pair classes from Fig. 3.",
               "Fill a symmetric motif-by-motif matrix per task using C_effective.",
               "Mask classes that are not direction-supported under the current FDR and effective-count filters."],
              fig3h_source)

def plot_full_tf_pair_matrix(fig3h_source, motifs):
    labels = [short_tf_name(m) for m in motifs]
    fig, axes = plt.subplots(1, 3, figsize=(9.2, 2.8), gridspec_kw={"width_ratios": [1, 1, 1]})
    cmap = mpl.colormaps["RdBu_r"].copy()
    cmap.set_bad("#EDEDED")
    for ax, task in zip(axes, TASK_ORDER):
        mat = pd.DataFrame(np.nan, index=motifs, columns=motifs, dtype=float)
        sub = fig3h_source[(fig3h_source.task == task) & (fig3h_source.direction_supported)]
        for _, r in sub.iterrows():
            mat.loc[r.tf1, r.tf2] = r.C_effective
            mat.loc[r.tf2, r.tf1] = r.C_effective
        im = ax.imshow(mat.to_numpy(float), cmap=cmap, vmin=-1, vmax=1, aspect="equal")
        ax.set_xticks(np.arange(len(motifs))); ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=5.4)
        ax.set_yticks(np.arange(len(motifs))); ax.set_yticklabels(labels if ax is axes[0] else [], fontsize=5.4)
        ax.set_title(task, color=TASK_COLOR[task], fontsize=7.2)
        ax.tick_params(length=0)
        for x in np.arange(len(motifs) + 1) - 0.5:
            ax.axvline(x, color="white", lw=0.35)
            ax.axhline(x, color="white", lw=0.35)
        for spine in ax.spines.values():
            spine.set_visible(False)
        if ax is axes[0]:
            add_panel_label(ax, "h", x=-0.16, y=1.04)
    cax = fig.add_axes([0.91, 0.22, 0.018, 0.56])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label("C_effective", fontsize=6)
    cbar.ax.tick_params(labelsize=5.6, length=2)
    fig.subplots_adjust(left=.065, right=.86, bottom=.27, top=.84, wspace=.20)
    return fig

fig = plot_full_tf_pair_matrix(fig3h_source, fig3h_motifs)
save_figure(fig, "Fig3h_complete_tf_pair_nonadditivity_matrix")

## Fig. 4 — motif-pair distance with fixed-flank context

Distance is not interpreted alone. A focal pair can have additional motif centers nearby, and long-distance bins can be noisy. This figure prints bin support and stratifies each distance bin by external 50-bp flank context.

In [ ]:
def build_distance_context():
    rows = []
    for t in TASKS:
        pairs, locs = raw[t.label]["motif_combi_isa"], raw[t.label]["motif_locs"]
        centers = {r: ((g.start_rel + g.end_rel)/2).to_numpy(float) for r, g in locs.groupby("region")}
        inter = interaction_col(t)
        for p in pairs.itertuples(index=False):
            left, right = min(p.start1_rel, p.start2_rel), max(p.end1_rel, p.end2_rel)
            if left < FLANK_WIDTH_BP or right + FLANK_WIDTH_BP > SEQUENCE_WINDOW_END: continue
            c = centers.get(p.region)
            if c is None: continue
            flank = ((c >= left-FLANK_WIDTH_BP) & (c < left)) | ((c > right) & (c <= right+FLANK_WIDTH_BP))
            rows.append({"task": t.label, "context": "Flank-dense" if flank.sum() >= 1 else "Flank-sparse",
                         "distance_bin": int(np.floor(p.distance/DISTANCE_BIN_SIZE)*DISTANCE_BIN_SIZE), "interaction": float(getattr(p, inter))})
    inst = pd.DataFrame(rows)
    summary = inst.groupby(["task", "context", "distance_bin"], observed=True).agg(
        mean_interaction=("interaction", "mean"), median_abs_interaction=("interaction", lambda x: float(np.median(np.abs(x)))), n=("interaction", "size")).reset_index()
    summary["passes_n_filter"] = summary.n >= STABLE_DISTANCE_BIN_N
    return inst, summary

fig4_inst, fig4 = build_distance_context()
fig4_inst.to_csv(SOURCE_DIR / "fig4_distance_context_instances.csv", index=False)
fig4.to_csv(SOURCE_DIR / "fig4_distance_context_binned_summary.csv", index=False)
panel_context("Fig. 4", "a-d", "Distance with fixed-flank context",
              "Does motif-pair distance relate to interaction after accounting for additional motif centers near the focal pair?",
              ["Keep focal pairs with room for 50-bp external flanks.", "Classify Flank-dense vs Flank-sparse.", "Bin distance every 10 bp.", "Plot primary lines only when bin n >= 30."],
              fig4.sort_values(["task", "context", "distance_bin"]))
print("Unstable bins that are not used for primary line interpretation:")
display(fig4[~fig4.passes_n_filter].sort_values(["task", "context", "distance_bin"]).head(60))

In [ ]:
def plot_distance_schematic(ax):
    ax.set_xlim(0, 250); ax.set_ylim(0, 1); ax.hlines(.5, 0, 249, color="#B8B8B8", lw=2)
    ax.add_patch(plt.Rectangle((80,.38),18,.24,color=TASK_COLOR["CAGE"],alpha=.85)); ax.add_patch(plt.Rectangle((150,.38),18,.24,color=TASK_COLOR["CAGE"],alpha=.85))
    ax.annotate("", xy=(150,.72), xytext=(98,.72), arrowprops=dict(arrowstyle="<->", lw=.8, color=REAL_COLOR)); ax.text(124,.78,"distance",ha="center",fontsize=6)
    ax.add_patch(plt.Rectangle((30,.18),50,.16,color="#E8E8E8",ec="#767676",lw=.6)); ax.add_patch(plt.Rectangle((168,.18),50,.16,color="#E8E8E8",ec="#767676",lw=.6))
    ax.text(55,.12,"50-bp flank",ha="center",fontsize=6); ax.text(193,.12,"50-bp flank",ha="center",fontsize=6); ax.plot([45,205],[.5,.5],"o",color="#C9473D",ms=3)
    ax.set_xticks([]); ax.set_yticks([]); [s.set_visible(False) for s in ax.spines.values()]; ax.set_title("Distance grammar"); add_panel_label(ax, "a", x=-.02, y=.95)

def plot_fig4_task(ax, task, label):
    sub = fig4[(fig4.task == task) & fig4.passes_n_filter]
    for ctx in ["Flank-sparse", "Flank-dense"]:
        s = sub[sub.context == ctx].sort_values("distance_bin")
        ax.plot(s.distance_bin, s.mean_interaction, marker="o", ms=3.2, lw=1.2, color=CONTEXT_COLORS[ctx], label=ctx)
    ax.axhline(0, color="#999999", lw=.7, ls=":"); ax.set_title(task); ax.set_xlabel("Distance bin (bp)"); ax.set_ylabel("Mean interaction"); add_panel_label(ax, label)

for task, lab in zip(TASK_ORDER, ["b", "c", "d"]):
    panel_context("Fig. 4", lab, f"{task} distance trend", f"For {task}, which bins support the plotted distance trend?",
                  ["Print all bin support for this task.", "Only n >= 30 bins are plotted."], fig4[fig4.task == task].sort_values(["context", "distance_bin"]))

fig, ax = plt.subplots(figsize=(2.4, 2.0))
plot_distance_schematic(ax)
save_figure(fig, "Fig4a_distance_flank_context_schematic")

for task, lab in zip(TASK_ORDER, ["b", "c", "d"]):
    fig, ax = plt.subplots(figsize=(2.2, 2.0))
    plot_fig4_task(ax, task, lab)
    ax.legend(fontsize=6)
    save_figure(fig, f"Fig4{lab}_{task}_distance_context_trend")

fig, axes = plt.subplots(1, 4, figsize=(7.6, 2.25), gridspec_kw={"width_ratios": [1.15, 1, 1, 1]})
plot_distance_schematic(axes[0])
for ax, task, lab in zip(axes[1:], TASK_ORDER, ["b", "c", "d"]): plot_fig4_task(ax, task, lab)
axes[-1].legend(fontsize=6); fig.subplots_adjust(wspace=.5, bottom=.22, top=.82)
save_figure(fig, "Fig4_distance_with_flank_context")

# Support/count audit for Fig. 4. This is the place to check whether long-distance fluctuations are supported by enough instances.
panel_context("Extended Data Fig. 4", "support", "Distance-bin support counts",
              "Which distance bins have enough motif-pair instances to support interpretation of Fig. 4 trends?",
              ["Use the same binned table as Fig. 4.", "Plot n per distance bin by task and flank context.", "The dashed line marks the primary plotting threshold n >= 30."],
              fig4[["task", "context", "distance_bin", "n", "passes_n_filter"]].sort_values(["task", "context", "distance_bin"]))
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.1), sharey=True)
for ax, task in zip(axes, TASK_ORDER):
    sub = fig4[fig4.task == task]
    for ctx in ["Flank-sparse", "Flank-dense"]:
        s = sub[sub.context == ctx].sort_values("distance_bin")
        ax.plot(s.distance_bin, s.n, marker="o", ms=3, lw=1.1, color=CONTEXT_COLORS[ctx], label=ctx)
    ax.axhline(STABLE_DISTANCE_BIN_N, color="#999999", lw=.8, ls=":")
    ax.set_title(task); ax.set_xlabel("Distance bin (bp)")
    if ax is axes[0]: ax.set_ylabel("Pair instances per bin")
axes[-1].legend(fontsize=6)
fig.subplots_adjust(wspace=.35, bottom=.23, top=.82)
save_figure(fig, "ExtDataFig4_distance_bin_support_counts")

## Extended Data Fig. 5 — TF-pair class non-additivity versus median motif-pair distance

The distance trend in Fig. 4 is instance-level. This scatter plot asks a different question: after TF-pair class aggregation, do direction-supported TF-pair classes occupy distinct median-distance ranges? The x-axis is the class-level net non-additivity score `C_effective`, the y-axis is the median motif-pair distance among eligible instances for that TF-pair class, and point size denotes the number of effective null-exceeding instances. This is a class-level diagnostic, not a biochemical synergy classification.

In [ ]:
def build_class_distance_scatter_source():
    df = fig3_classes.copy()
    df = df[np.isfinite(df["C_effective"]) & np.isfinite(df["median_distance"])].copy()
    df["plot_class"] = "Not direction-supported"
    df.loc[df.direction_supported & (df.C_effective < 0), "plot_class"] = "Negative direction-supported"
    df.loc[df.direction_supported & (df.C_effective > 0), "plot_class"] = "Positive direction-supported"
    df["pair_label"] = df["tf_pair"].map(short_pair_label)
    df["point_size"] = 10 + np.sqrt(df["n_effective_instances"].clip(lower=1)) * 3.2
    return df[[
        "task", "tf_pair", "pair_label", "C_effective", "median_distance",
        "n_eligible_instances", "n_effective_instances", "mw_q",
        "direction_supported", "direction_class", "plot_class", "point_size"
    ]]

fig4e_source = build_class_distance_scatter_source()
fig4e_source.to_csv(SOURCE_DIR / "extended_fig5_tf_pair_class_distance_nonadditivity_scatter_source.csv", index=False)

def build_class_distance_scatter_stats(df):
    rows = []
    for task, g in df.groupby("task"):
        rho_all, p_all = spearmanr(g["C_effective"], g["median_distance"])
        ds = g[g["direction_supported"]].copy()
        rho_ds, p_ds = spearmanr(ds["C_effective"], ds["median_distance"]) if len(ds) >= 3 else (np.nan, np.nan)
        rows.append({
            "task": task,
            "tested_classes_n": len(g),
            "direction_supported_classes_n": len(ds),
            "spearman_rho_all_tested": rho_all,
            "spearman_p_all_tested": p_all,
            "spearman_rho_direction_supported": rho_ds,
            "spearman_p_direction_supported": p_ds,
        })
    return pd.DataFrame(rows).set_index("task").loc[TASK_ORDER].reset_index()

fig4e_stats = build_class_distance_scatter_stats(fig4e_source)
fig4e_stats.to_csv(SOURCE_DIR / "extended_fig5_tf_pair_class_distance_nonadditivity_scatter_stats.csv", index=False)
panel_context("Extended Data Fig. 5", "", "Class-level non-additivity versus median motif-pair distance",
              "Do direction-supported TF-pair classes occupy distinct median-distance ranges?",
              ["Use the TF-pair class table from Fig. 3.",
               "Keep classes with finite C_effective and finite median motif-pair distance.",
               "Plot C_effective against median distance separately for CAGE, DEV and HK.",
               "Encode direction-supported positive/negative classes by color and use grey for classes not retained by the final filters.",
               "Scale point area by n_effective_instances.",
               "Report Spearman correlation between C_effective and median distance for all tested classes and for direction-supported classes."],
              fig4e_source.sort_values(["task", "direction_supported", "C_effective"], ascending=[True, False, True]))
display(fig4e_stats)

def plot_class_distance_scatter(df, stats):
    from matplotlib.lines import Line2D
    colors = {
        "Not direction-supported": "#B8B8B8",
        "Negative direction-supported": "#5B7FCA",
        "Positive direction-supported": "#D9544D",
    }
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.45), sharey=True)
    for ax, task in zip(axes, TASK_ORDER):
        sub = df[df.task == task].copy()
        for cls in ["Not direction-supported", "Negative direction-supported", "Positive direction-supported"]:
            g = sub[sub.plot_class == cls]
            if len(g) == 0:
                continue
            ax.scatter(g.C_effective, g.median_distance, s=g.point_size,
                       c=colors[cls], edgecolor="white", linewidth=0.35,
                       alpha=0.45 if cls == "Not direction-supported" else 0.88,
                       label=cls)
        ax.axvline(0, color="#999999", lw=0.8, ls=":")
        ax.set_title(task, color=TASK_COLOR[task], fontsize=7.2)
        ax.set_xlabel("TF-pair net non-additivity score")
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(0, float(df.median_distance.max()) + 12)
        ax.tick_params(labelsize=6)
        st = stats[stats.task == task].iloc[0]
        ax.text(0.03, 0.95,
                "n={}\nSpearman rho={:.2f}\nP={}".format(
                    int(st.tested_classes_n),
                    st.spearman_rho_all_tested,
                    format_p(st.spearman_p_all_tested)
                ),
                transform=ax.transAxes, ha="left", va="top", fontsize=5.3, color="#444444")
        if ax is axes[0]:
            ax.set_ylabel("Median motif-pair distance (bp)")
            add_panel_label(ax, "a", x=-0.18, y=1.05)
    legend_specs = [
        ("Negative direction-supported", "Negative"),
        ("Positive direction-supported", "Positive"),
        ("Not direction-supported", "Not retained"),
    ]
    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="", markersize=5.5,
               markerfacecolor=colors[k], markeredgecolor="white", markeredgewidth=0.4,
               alpha=0.6 if k == "Not direction-supported" else 0.9)
        for k, _ in legend_specs
    ]
    legend_labels = [v for _, v in legend_specs]
    fig.legend(legend_handles, legend_labels, loc="upper center", bbox_to_anchor=(0.52, 1.04),
               ncol=3, fontsize=6, frameon=False, columnspacing=1.0, handletextpad=0.4)
    fig.text(0.985, 0.03, "Point area scales with n_eff", ha="right", va="center", fontsize=5.4, color="#555555")
    fig.subplots_adjust(left=.08, right=.985, bottom=.24, top=.78, wspace=.22)
    return fig

fig = plot_class_distance_scatter(fig4e_source, fig4e_stats)
save_figure(fig, "ExtDataFig5_tf_pair_class_distance_nonadditivity_scatter")

## Fig. 5 — CAGE-associated motif-position overlap architecture

Fig. 5 is an overlap/grammar figure. It does not reuse full-task ISA null comparisons. It asks how CAGE motif-called regions overlap DEV/HK at the region and motif-position interval levels.

In [ ]:
def interval_overlap(a, b, same_tf=False):
    m = a[["region","start_rel","end_rel","tf"]].merge(b[["region","start_rel","end_rel","tf"]], on="region", suffixes=("_a","_b"))
    if m.empty: return m
    ok = (m[["end_rel_a","end_rel_b"]].min(axis=1) - m[["start_rel_a","start_rel_b"]].max(axis=1)) > 0
    if same_tf: ok = ok & (m.tf_a.astype(str) == m.tf_b.astype(str))
    return m.loc[ok].drop_duplicates()

loc = {t.label: raw[t.label]["motif_locs"].copy() for t in TASKS}
sets = {k: set(v.region.astype(str)) for k, v in loc.items()}
all_regions = sorted(set().union(*sets.values()))
membership = pd.DataFrame([{"region": r, "CAGE": r in sets["CAGE"], "DEV": r in sets["DEV"], "HK": r in sets["HK"]} for r in all_regions])
membership["region_state"] = membership.apply(lambda x: "".join([s for s in ["CAGE","DEV","HK"] if x[s]]) or "none", axis=1)
cd, ch, dh = interval_overlap(loc["CAGE"], loc["DEV"]), interval_overlap(loc["CAGE"], loc["HK"]), interval_overlap(loc["DEV"], loc["HK"])
cd_same, ch_same = interval_overlap(loc["CAGE"], loc["DEV"], True), interval_overlap(loc["CAGE"], loc["HK"], True)
cd_regions, ch_regions = set(cd.region.astype(str)), set(ch.region.astype(str))
cage_classes = pd.DataFrame([{"region": r, "cage_overlap_class": ("CAGE-with-DEV-and-HK" if r in cd_regions and r in ch_regions else "CAGE-with-DEV" if r in cd_regions else "CAGE-with-HK" if r in ch_regions else "CAGE-only")} for r in sorted(sets["CAGE"])])
overlap_summary = pd.DataFrame([
    {"comparison": "CAGE-DEV motif-position overlap", "regions": cd.region.nunique(), "motif_interval_pairs": len(cd), "same_tf_interval_pairs": len(cd_same)},
    {"comparison": "CAGE-HK motif-position overlap", "regions": ch.region.nunique(), "motif_interval_pairs": len(ch), "same_tf_interval_pairs": len(ch_same)},
    {"comparison": "DEV-HK motif-position overlap", "regions": dh.region.nunique(), "motif_interval_pairs": len(dh), "same_tf_interval_pairs": np.nan},
])
membership.to_csv(SOURCE_DIR / "fig5_region_level_membership_all_states.csv", index=False)
cage_classes.to_csv(SOURCE_DIR / "fig5_cage_position_overlap_classes.csv", index=False)
overlap_summary.to_csv(SOURCE_DIR / "fig5_position_overlap_summary.csv", index=False)
state_counts = membership.region_state.value_counts().rename_axis("region_state").reset_index(name="n_regions")
class_counts = cage_classes.cage_overlap_class.value_counts().rename_axis("cage_overlap_class").reset_index(name="n_regions")
panel_context("Fig. 5", "a", "All region-level CAGE/DEV/HK states", "How many motif-called regions are present in each task combination?", ["Use motif_locs region membership only."], state_counts)
panel_context("Fig. 5", "b", "CAGE-associated motif-position overlap classes", "Among CAGE regions, which have motif-position interval overlap with DEV and/or HK?", ["Use within-region interval overlap.", "Assign CAGE-only, CAGE-with-DEV, CAGE-with-HK and CAGE-with-DEV-and-HK."], class_counts)
panel_context("Fig. 5", "c", "Overlap audit", "How many interval overlaps and same-TF overlaps define these classes?", ["Count interval-pair overlaps and same-TF overlaps for CAGE-DEV/CAGE-HK."], overlap_summary)

def build_cage_identity_overlap_source():
    frames = []
    for comparison, df in [("CAGE-DEV", cd), ("CAGE-HK", ch)]:
        if df.empty:
            continue
        tmp = df.copy()
        tmp["cage_tf"] = tmp["tf_a"].astype(str)
        tmp["other_tf"] = tmp["tf_b"].astype(str)
        counts = tmp.groupby(["cage_tf", "other_tf"], observed=True).agg(
            interval_pairs=("region", "size"),
            regions=("region", "nunique"),
        ).reset_index()
        counts["comparison"] = comparison
        frames.append(counts)
    source = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["comparison", "cage_tf", "other_tf", "interval_pairs", "regions"])
    return source

fig5d_source = build_cage_identity_overlap_source()
fig5d_source.to_csv(SOURCE_DIR / "fig5d_cage_associated_motif_identity_overlap_matrix_source.csv", index=False)
panel_context("Fig. 5", "d", "CAGE-to-DEV/HK motif-position overlap identity matrices",
              "Which motif identities drive CAGE-with-DEV and CAGE-with-HK motif-position overlap?",
              ["Use motif-position interval overlaps from CAGE-DEV and CAGE-HK.",
               "Group overlaps by CAGE motif identity and the paired DEV or HK motif identity.",
               "Plot interval-pair counts as a log-scaled matrix."],
              fig5d_source.sort_values(["comparison", "interval_pairs"], ascending=[True, False]).head(60))

In [ ]:
def plot_fig5a(ax):
    order = [x for x in ["CAGE", "DEV", "HK", "CAGEDEV", "CAGEHK", "DEVHK", "CAGEDEVHK"] if x in set(state_counts.region_state)]
    tmp = state_counts.set_index("region_state").reindex(order).dropna().reset_index()
    state_label = {
        "CAGE": "CAGE",
        "DEV": "DEV",
        "HK": "HK",
        "CAGEDEV": "CAGE-DEV",
        "CAGEHK": "CAGE-HK",
        "DEVHK": "DEV-HK",
        "CAGEDEVHK": "CAGE-DEV-HK",
    }
    xlabels = [state_label.get(x, x) for x in tmp.region_state]
    ax.bar(np.arange(len(tmp)), tmp.n_regions, color="#D9D9D9", edgecolor="#4D4D4D", linewidth=.6)
    ax.set_xticks(np.arange(len(tmp))); ax.set_xticklabels(xlabels, rotation=45, ha="right")
    ax.set_ylabel("Regions"); ax.set_title("Region-level states"); add_panel_label(ax, "a")

def plot_fig5b(ax):
    order = ["CAGE-only", "CAGE-with-DEV", "CAGE-with-HK", "CAGE-with-DEV-and-HK"]
    tmp = class_counts.set_index("cage_overlap_class").reindex(order).fillna(0).reset_index()
    ax.barh(np.arange(len(tmp)), tmp.n_regions, color=["#D9D9D9", TASK_COLOR["DEV"], TASK_COLOR["HK"], "#4D4D4D"], edgecolor="#4D4D4D", linewidth=.6)
    ax.set_yticks(np.arange(len(tmp))); ax.set_yticklabels(tmp.cage_overlap_class, fontsize=6); ax.invert_yaxis()
    ax.set_xlabel("CAGE regions"); ax.set_title("CAGE-associated classes"); add_panel_label(ax, "b")

centers = []
for t in TASKS:
    d = raw[t.label]["motif_locs"].copy(); d["center_rel"] = (d.start_rel + d.end_rel)/2; d["task"] = t.label; centers.append(d[["task","center_rel"]])
centers = pd.concat(centers, ignore_index=True); centers.to_csv(SOURCE_DIR / "fig5c_motif_center_density_source.csv", index=False)

def plot_fig5c(ax):
    for task in TASK_ORDER:
        sns.kdeplot(centers.loc[centers.task == task, "center_rel"], ax=ax, color=TASK_COLOR[task], lw=1.2, label=task)
    ax.set_xlabel("Motif centre in 250-bp region"); ax.set_ylabel("Density"); ax.set_title("Motif-position density"); ax.legend(fontsize=6); add_panel_label(ax, "c")

def plot_fig5d(fig5d_source, axes=None):
    created = axes is None
    if created:
        fig, axes = plt.subplots(1, 2, figsize=(6.1, 2.45), sharey=True)
    motifs = sorted(set(fig5d_source.cage_tf.astype(str)).union(set(fig5d_source.other_tf.astype(str))))
    labels = [short_tf_name(m) for m in motifs]
    vmax = np.log10(fig5d_source.interval_pairs.max() + 1) if len(fig5d_source) else 1
    for ax, comparison, color, xlab in zip(
        axes,
        ["CAGE-DEV", "CAGE-HK"],
        [TASK_COLOR["DEV"], TASK_COLOR["HK"]],
        ["DEV motif identity", "HK motif identity"],
    ):
        sub = fig5d_source[fig5d_source.comparison == comparison]
        mat = pd.DataFrame(0.0, index=motifs, columns=motifs)
        for _, r in sub.iterrows():
            mat.loc[r.cage_tf, r.other_tf] = np.log10(float(r.interval_pairs) + 1.0)
        im = ax.imshow(mat.to_numpy(float), cmap="Greys", vmin=0, vmax=vmax, aspect="equal")
        ax.set_xticks(np.arange(len(motifs))); ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=5.4)
        ax.set_yticks(np.arange(len(motifs))); ax.set_yticklabels(labels if ax is axes[0] else [], fontsize=5.4)
        ax.set_title(comparison, color=color, fontsize=7.2)
        ax.tick_params(length=0)
        for x in np.arange(len(motifs) + 1) - 0.5:
            ax.axvline(x, color="white", lw=0.35)
            ax.axhline(x, color="white", lw=0.35)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_xlabel(xlab, fontsize=6.4)
    axes[0].set_ylabel("CAGE motif identity", fontsize=6.4)
    add_panel_label(axes[0], "d", x=-0.18, y=1.04)
    if created:
        fig.subplots_adjust(left=.12, right=.82, bottom=.28, top=.82, wspace=.28)
        cax = fig.add_axes([0.88, 0.25, 0.018, 0.52])
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label("Interval-overlap pairs\n(log10 count + 1)", fontsize=6)
        cbar.ax.tick_params(labelsize=5.6, length=2)
        return fig
    return im

fig, ax = plt.subplots(figsize=(2.2, 2.1))
plot_fig5a(ax)
save_figure(fig, "Fig5a_region_level_membership_states")

fig, ax = plt.subplots(figsize=(2.7, 2.1))
plot_fig5b(ax)
save_figure(fig, "Fig5b_cage_position_overlap_classes")

fig, ax = plt.subplots(figsize=(2.5, 2.1))
plot_fig5c(ax)
save_figure(fig, "Fig5c_motif_position_density")

fig = plot_fig5d(fig5d_source)
save_figure(fig, "Fig5d_cage_dev_hk_motif_position_overlap_identity_matrices")

fig = plt.figure(figsize=(7.6, 4.55))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.1], width_ratios=[0.95, 1.35, 1.15], hspace=.65, wspace=.75)
ax_a = fig.add_subplot(gs[0, 0]); ax_b = fig.add_subplot(gs[0, 1]); ax_c = fig.add_subplot(gs[0, 2])
ax_d1 = fig.add_subplot(gs[1, 0]); ax_d2 = fig.add_subplot(gs[1, 1], sharey=ax_d1)
ax_blank = fig.add_subplot(gs[1, 2]); ax_blank.axis("off")
plot_fig5a(ax_a)
plot_fig5b(ax_b)
plot_fig5c(ax_c)
im = plot_fig5d(fig5d_source, axes=[ax_d1, ax_d2])
cax = fig.add_axes([0.89, 0.18, 0.018, 0.25])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Interval-overlap pairs\n(log10 count + 1)", fontsize=6)
cbar.ax.tick_params(labelsize=5.6, length=2)
fig.subplots_adjust(left=.08, right=.86, bottom=.15, top=.9)
save_figure(fig, "Fig5_cage_associated_motif_position_overlap")

## Extended audit and output manifest

These checks should be inspected before treating any class-level claim as final. They show whether `C_all` and `C_effective` disagree, and list all generated figure/source-data files.

In [ ]:
sens = fig3_classes.copy()
sens["C_all_sign"] = sens.C_all.map(score_sign); sens["C_effective_sign"] = sens.C_effective.map(score_sign)
sens["sign_changes_between_all_and_effective"] = sens.C_all_sign != sens.C_effective_sign
summary = sens.groupby("task").agg(tested_classes=("tf_pair","size"), q_supported=("passes_q","sum"), direction_supported=("direction_supported","sum"), sign_changes=("sign_changes_between_all_and_effective","sum")).reset_index()
summary.to_csv(SOURCE_DIR / "extended_tf_pair_class_sensitivity_summary.csv", index=False)
panel_context("Extended Data", "1", "C_all versus C_effective sensitivity",
              "Does the TF-pair sign depend on using only null-exceeding effective interactions?",
              ["Compute C_all from all eligible interactions.", "Compute C_effective from only interactions outside null thresholds.", "Flag sign changes."], summary)
display(sens[sens.sign_changes_between_all_and_effective].sort_values(["task","mw_q"]).head(50))

manifest = []
for p in sorted(FIG_DIR.glob("*")):
    row = {"file": str(p), "suffix": p.suffix, "bytes": p.stat().st_size}
    if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
        with Image.open(p) as im: row["width_px"], row["height_px"] = im.size
    manifest.append(row)
manifest = pd.DataFrame(manifest)
manifest.to_csv(OUT_ROOT / "figure_output_manifest.csv", index=False)
display(manifest)
print("Source data files:")
for p in sorted(SOURCE_DIR.glob("*.csv")): print("-", p.name)

## Figure legend draft

**Fig. 1 | Full-task ISA output scale and task-specific null calibration.** a, Counts of single-motif ISA records and motif-pair interaction records in the complete CAGE, DEV and HK ISA outputs. Single-motif records correspond to one motif interval perturbed at a time and do not imply that the sequence contains only one motif. Motif-pair records correspond to two motif intervals evaluated for non-additive response in the same region. b, Median single-motif ISA for motif-called intervals compared with the corresponding task-specific single-null distribution. The single-null distribution is generated from length-matched non-motif pseudo-sites scored with the same single-ablation ISA calculation. c, Median motif-pair interaction compared with the corresponding task-specific pair-null distribution. The pair-null distribution is generated from distance-matched non-motif pseudo-site pairs scored with the same pair-ablation interaction calculation. CAGE and DEV use track 0; HK uses track 1. Null thresholds and loaded null counts are reported in `source_data/null_method_summary.csv`; figure source data are generated in `source_data/fig1_full_task_scale_and_null_source.csv`.

**Fig. 2 | Full distribution comparison and motif-identity resolution of motif-called responses.** a,b, Kernel-density estimates show single-motif ISA score distributions in the top row and motif-pair interaction score distributions in the bottom row for each task. The single-motif panels use each task's length-matched single-null table, whereas the pair-interaction panels use each task's distance-matched pair-null table. Each panel reports motif-called and null sample sizes, medians and a two-sided Mann-Whitney U test. c, Motif identity-level median single-motif ISA after subtracting the corresponding task-specific single-motif null median. Numbers inside cells denote single-motif record counts for each motif identity and task. Panel c is a single-motif ISA panel and is not a motif-pair interaction analysis. Source data are generated in `source_data/fig2_real_vs_null_distribution_summary.csv` and `source_data/fig2c_single_motif_identity_null_adjusted_source.csv`.

**Fig. 3 | From motif-pair instances to direction-supported TF-pair non-additivity.** a, Instance-level gate composition among all valid motif-pair interaction instances. `Single-ISA gate only` denotes instances where both individual motif perturbations exceed the task-specific length-matched single-null threshold but the pair interaction does not exceed the task-specific distance-matched pair-null threshold; `Pair gate only` denotes the converse; `Both gates` denotes instances satisfying both criteria. b, Class-level filtering from raw TF-pair classes to final direction-supported classes. Classes are retained after eligibility filtering, a minimum of 10 eligible motif-pair instances, task-wise Mann-Whitney testing against the pair-interaction null distribution, BH-FDR correction at q<0.10, and a minimum of 10 effective null-exceeding interactions. c, All raw motif-pair interaction instances classified as negative, near-null or positive relative to task-specific pair-null thresholds. d, Empirical cumulative distributions of the net non-additivity score for direction-supported TF-pair classes. e, Counts of direction-supported TF-pair classes after task-wise FDR control and effective-count filtering. f, FDR-threshold sensitivity comparing discovery-level q<0.10 with stringent q<0.05. N/P denotes negative/positive net non-additivity classes. g, Motif-pair-resolution heatmap for the union of the top seven direction-supported TF-pair classes per task, ranked by absolute `C_effective`. The upper heatmap shows task-wise `C_effective`; grey cells indicate that the selected TF pair is not direction-supported in that task. The middle strip marks direction-supported membership in CAGE, DEV and/or HK. The lower `n_eff` strip shows the total number of effective null-exceeding instances supporting each selected TF-pair class across displayed tasks, with low-support columns labelled directly. h, Complete motif-by-motif TF-pair matrices for all tested classes in each task. Colored cells indicate direction-supported TF-pair classes and grey cells indicate classes not retained by the current FDR and effective-count filters. Source data are generated in `source_data/fig3*.csv`.

**Fig. 4 | Distance-associated pair interaction with fixed-flank context.** a, Schematic of focal pair distance and external 50-bp flank annotation. b-d, Mean pair interaction across 10-bp distance bins for CAGE, DEV and HK, stratified by whether additional motif centers occur in the fixed external flanks. Only bins with at least 30 pair instances are shown in the primary lines; full support counts are printed and saved. Source data are generated in `source_data/fig4_distance_context_binned_summary.csv`.

**Extended Data Fig. 4 | Distance-bin support for Fig. 4.** Counts of motif-pair instances per 10-bp distance bin are shown for each task and flank context. The dashed line marks the minimum bin support threshold used for the primary Fig. 4 trend lines. Source data are generated in `source_data/fig4_distance_context_binned_summary.csv`.

**Extended Data Fig. 5 | TF-pair class non-additivity and median motif-pair distance.** Class-level scatter plots show `C_effective` versus median motif-pair distance for tested TF-pair classes in CAGE, DEV and HK. Blue and red points denote direction-supported negative and positive classes after the Fig. 3 class-level filters; grey points denote tested classes not retained as direction-supported. Point area scales with the number of effective null-exceeding instances (`n_eff`). Spearman rho and P values are calculated across all tested TF-pair classes within each task. This is a class-level diagnostic linking Fig. 3 and Fig. 4 and is not a biochemical synergy classification. Source data are generated in `source_data/extended_fig5_tf_pair_class_distance_nonadditivity_scatter_source.csv` and `source_data/extended_fig5_tf_pair_class_distance_nonadditivity_scatter_stats.csv`.

**Fig. 5 | CAGE-associated motif-position overlap architecture.** a, Region-level motif-call membership states across CAGE, DEV and HK. b, CAGE motif-called regions stratified by motif-position interval overlap with DEV and/or HK. c, Relative motif-centre distributions across the 250-bp regions. d, Motif-position overlap identity matrices for CAGE-DEV and CAGE-HK. Rows are CAGE motif identities. Columns are DEV motif identities in the CAGE-DEV matrix and HK motif identities in the CAGE-HK matrix. Intensity denotes the number of overlapping motif-interval pairs, shown as log10(count + 1). This figure describes motif-position architecture and is not a full-task null-calibrated ISA comparison. Source data are generated in `source_data/fig5*_*.csv`.